### This script is used to predict RFR normalized concentrations of rOC, DIN and SRP using the optimized models:
### Thus, 5 files containing RFR normalized concentrations of roC, DIN and SRP are created (for each target variable 5 different models were fitted (using different train_test_splits))


# Load required modules: 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms
from sklearn import metrics
from numpy import mean
from numpy import std
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
import time
import os
import ast
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import os
from sklearn.preprocessing import PowerTransformer
import pdb
import time

In [3]:
# Loading Data
data = pd.read_csv('../../../output_data/input_ML_learning/median_cnp_export_abs_conc_and_rfr_and_basin_feat.csv').drop(columns='Unnamed: 0')
optimized_parameter_sets = pd.read_csv('../../../output_data/ML_analysis/hyperparameter_settings/optimized_model_parms_for_importance_calculation.csv', sep = ';', dtype={'parameters': str})
# Convert the 'parameters' column to dictionaries
optimized_parameter_sets['parameters'] = optimized_parameter_sets['parameters'].apply(ast.literal_eval)
# Define the directory path

In [4]:
def create_input_ML(data, target_var):

    # create array with exported nutrient values, which are our target values, we want to predict 
    target = np.array(data[target_var])
    # remove the labels from the features df:
    features = data.drop(['n_imbal', 'p_imbal', 'c_imbal', 'median_DOC_TOC_bioav', 'median_DIN', 'median_SRP'], axis = 1)
    #features = nutrient_export_data.filter(items=selected_features)

    #select columns by index: 
    features= features.iloc[:,0:43]
    features.columns

    # save feature list for later use:
    feature_list = list(features.columns)

    # convert to numpy array:
    #features = np.array(features)

    return target, features, feature_list

In [5]:
#select model type: 'gbr' or 'lightgbm'
model_type = 'gbr'
#set random_state:
run_number = [1,2,3,4,5]
random_state_split = [2, 19, 44, 33, 21]
random_state_regressor = [4, 9, 19, 31, 41]
random_state_feature_importances = [1, 17, 45, 22, 12]

### Now fit for all five splits the models:

In [6]:
# select target_var: n_imbal, c_imbal, p_imbal, median_DIN, median_SRP or median_DOC_TOC_bioav
#target_var = 'p_imbal'

# create empty data lists to save data:
# test_features_runs
t0 = time.time()
df_outputs = []
df_all_outputs = []
df_metrics = []
transf_bc = {}
lambda_values = {}
pt_dict = {}
train_targets_splits = {}
train_features_splits = {}
test_features_splits = {}

models_splits = {}
boxcox = True
stratified_sampling = False
for target_var in ['p_imbal','c_imbal','n_imbal']:  
    train_targets_splits[target_var] = [] 
    train_features_splits[target_var] = []
    lambda_values[target_var] = {}
    pt_dict[target_var] = {}
    target, features, feature_list = create_input_ML(
        data = data, target_var = target_var)
    # extract best parameters: squeeze() funtion is required, to convert Series object to one value (result of loc operation is always a pandas series, even if output is only one value):
    best_parms =  optimized_parameter_sets.loc[
        (optimized_parameter_sets['target'] == target_var) & (optimized_parameter_sets['model'] == model_type),
        'parameters'].squeeze()
    transf_bc[target_var] = []
    test_features_splits[target_var] = []
    r2_train_splits = []
    r2_test_splits = []
    models_splits[target_var] = []
    pred_test_bc_df = pd.DataFrame()
    pred_test_original_scale_df = pd.DataFrame()
    obs_test_bc_df = pd.DataFrame()
    obs_test_original_scale_df = pd.DataFrame()
    
    # loop to split the data into training and test data and to fit the model: using diffent random states:
    for i, (split, regressor, feature) in enumerate(zip(random_state_split, random_state_regressor, random_state_feature_importances)):       
        if stratified_sampling:
            num_quantiles = 20  # Adjust this to the number of quantiles you want
            target_binned = pd.qcut(target, q=num_quantiles, labels=False)
            train_features, test_features, train_targets, test_targets = train_test_split(
                features, target, test_size=0.20, random_state=split, stratify = target_binned)
        else:
            train_features, test_features, train_targets, test_targets = train_test_split(features, target, test_size=0.20, random_state=split)
        
        all_features = features.copy()
        train_IDs = train_features['HYBAS_ID'].copy()
        test_IDs = test_features['HYBAS_ID'].copy()
        all_IDs = all_features['HYBAS_ID'].copy()
        train_features =train_features.drop('HYBAS_ID',axis=1)
        test_features = test_features.drop('HYBAS_ID',axis=1)
        all_features = all_features.copy().drop('HYBAS_ID', axis = 1)
        all_targets = target.copy()

            
        if len(train_targets.shape) == 1:
            train_targets_flat = np.reshape(train_targets, (-1, 1))
        if len(test_targets.shape) == 1:
            test_targets_flat = np.reshape(test_targets, (-1, 1))

        if len(all_targets.shape) == 1:
            all_targets_flat = np.reshape(all_targets, (-1, 1))
        if boxcox:
            pt = PowerTransformer(method='box-cox')
            # Reshape train_targets to 2D if it's 1D
            # Fit the transformer to the training data
            # This learns the best lambda value for each feature
            pt.fit(train_targets_flat)
            lambda_values[target_var][i] = pt.lambdas_
            pt_dict[target_var][i] = pt
            # Transform the training data using the learned lambda values
            train_targets_transformed_bc = pt.transform(train_targets_flat)
            # convert back to 1D format
            train_targets_transformed_bc = train_targets_transformed_bc.ravel()

            # Transform the test data using the same lambda values
            # Reshape test_targets to 2D if it's 1D
            test_targets_transformed_bc = pt.transform(test_targets_flat)
            test_targets_transformed_bc = test_targets_transformed_bc.ravel()
            train_targets_to_model = train_targets_transformed_bc

            all_targets_transformed_bc = pt.transform(all_targets_flat)
            all_targets_transformed_bc = all_targets_transformed_bc.ravel()

        else:
            train_targets_to_model = train_targets_flat
        
        # save test_features to list:
        test_features_splits[target_var].append(test_features)
        train_targets_splits[target_var].append(train_targets_transformed_bc)
        train_features_splits[target_var].append(train_features)

        # Fit the model:   
        model = GradientBoostingRegressor(**best_parms,  random_state = regressor, validation_fraction = 0.1, n_iter_no_change = 10)
        model.fit(train_features, train_targets_to_model)
        # save the model to a list:
        models_splits[target_var].append(model)

        # make predictions on train- and test-features:
        pred_train = model.predict(train_features)
        pred_test = model.predict(test_features)
        pred_test_bc = pred_test.reshape(-1, 1)

        pred_all_bc = model.predict(all_features).reshape(-1,1)
        
        obs_test_original_scale = test_targets.reshape(-1, 1)
        df = pd.DataFrame()
        df_all = pd.DataFrame()
        metrics = pd.DataFrame()
        if boxcox:
            # transform back to original scale:
            pred_train_original_scale = pt.inverse_transform(pred_train.reshape(-1, 1))
            pred_test_original_scale = pt.inverse_transform(pred_test.reshape(-1, 1))
            pred_all_original_scale = pt.inverse_transform(pred_all_bc.reshape(-1,1))

            obs_test_bc = test_targets_transformed_bc.reshape(-1, 1)
            df['obs_test_norm'] = obs_test_bc.flatten().copy()
            df['preds_norm'] = pred_test_bc.flatten().copy()
            r2_train = r2_score(train_targets_transformed_bc, pred_train)
            r2_test = r2_score(test_targets_transformed_bc, pred_test)
            print(f"R2 Test boxcox {target_var,i}: {r2_test}")
            metrics.loc[0,'R2_test_boxcox'] = r2_test
        else:
            pred_test_original_scale = pred_test_bc

        df['obs_test'] = obs_test_original_scale.flatten().copy() 
        df['preds_test'] = pred_test_original_scale.flatten().copy()
        df['nrand'] = i
        df['comp'] = target_var
        df['station'] = test_IDs.values
        df_outputs.append(df.copy())

        df_all['obs_box_cox'] = all_targets_transformed_bc.flatten().copy()
        df_all['obs'] = all_targets_flat.copy()
        df_all['pred_box_cox'] = pred_all_bc.flatten().copy()
        df_all['preds'] = pred_all_original_scale.flatten().copy()
        df_all['nrand'] = i
        df_all['comp'] = target_var
        df_all['HYBAS_ID'] = all_IDs.values
        df_all_outputs.append(df_all.copy())


        r2_train_orig = r2_score(train_targets, pred_train_original_scale)
        r2_test_orig = r2_score(test_targets, pred_test_original_scale)
        #print(f"R2 Train boxcox {target_var,i}: {r2_train}")
        #print(f"R2 Train original scale {target_var,i}: {r2_train_orig}")
        print(f"R2 Test original scale {target_var,i}: {r2_test_orig}")
        metrics.loc[0,'R2_test'] = r2_test_orig
        metrics.loc[0,'nrand'] = i
        metrics.loc[0,'comp'] = target_var
        df_metrics.append(metrics.copy())
        #r2_train_splits.append(r2_train)
        #r2_test_splits.append(r2_test)
        
    print('Elapsed time',time.time()-t0)
df_outputs = pd.concat(df_outputs,axis=0)
df_all_outputs = pd.concat(df_all_outputs, axis = 0)

R2 Test boxcox ('p_imbal', 0): 0.384237053039312
R2 Test original scale ('p_imbal', 0): 0.3333361091884026
R2 Test boxcox ('p_imbal', 1): 0.3771722450259485
R2 Test original scale ('p_imbal', 1): 0.33054802072330913
R2 Test boxcox ('p_imbal', 2): 0.38586426485382574
R2 Test original scale ('p_imbal', 2): 0.3289651247125325
R2 Test boxcox ('p_imbal', 3): 0.33551627539187723
R2 Test original scale ('p_imbal', 3): 0.24340957989220824
R2 Test boxcox ('p_imbal', 4): 0.3699119975296674
R2 Test original scale ('p_imbal', 4): 0.3265525715918717
Elapsed time 35.997798442840576
R2 Test boxcox ('c_imbal', 0): 0.6540695670853867
R2 Test original scale ('c_imbal', 0): 0.4494863396029237
R2 Test boxcox ('c_imbal', 1): 0.5724326466072911
R2 Test original scale ('c_imbal', 1): 0.45053444381988095
R2 Test boxcox ('c_imbal', 2): 0.6018280249981082
R2 Test original scale ('c_imbal', 2): 0.38333802213534773
R2 Test boxcox ('c_imbal', 3): 0.5628681068630688
R2 Test original scale ('c_imbal', 3): 0.43120888

In [8]:
df_all_outputs

,obs_box_cox,obs,pred_box_cox,preds,nrand,comp,HYBAS_ID
0,1.433749,0.469480,1.217154,0.399101,0,p_imbal,1121145450
1,1.249854,0.409110,1.062716,0.354605,0,p_imbal,1121146530
2,0.533476,0.232677,0.439693,0.215335,0,p_imbal,1121150540
3,2.309339,0.872547,1.764749,0.597400,0,p_imbal,1121151290
4,1.905221,0.660061,1.219135,0.399701,0,p_imbal,1121151840
...,...,...,...,...,...,...,...
3491,-1.647245,0.379972,-1.333987,0.485362,4,n_imbal,8120253120
3492,0.787938,0.882384,-1.202849,0.521761,4,n_imbal,8120256970
3493,0.198969,0.797126,-0.876187,0.600723,4,n_imbal,8120280400
3494,-1.870262,0.272594,-1.446920,0.450969,4,n_imbal,8120295370


In [10]:
# define the output folder path:

output_folder = '../../../output_data/ML_analysis/predicted_data'

# Check if the folder exists, if not, create it:
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [11]:
df_all_outputs.to_csv('../../../output_data/ML_analysis/predicted_data/predicted_cnp_all_catchments_and_models.csv')

In [12]:
features

,HYBAS_ID,UP_AREA,twi90,slp_dg_uav,for_pc_use,crp_pc_use,pst_pc_use,ppd_pk_uav,run_mm_syr,inu_pc_umn,...,pac_pc_use,cly_pc_uav,slt_pc_uav,snd_pc_uav,soc_th_uav,swc_pc_uyr,kar_pc_use,ero_kh_uav,gdp_ud_usu,hdi_ix_sav
0,1121145450,20609.9,8.826505,48,42,5,22,217.847,102,2,...,20,32,23,44,27,49,0,11117,1.175299e+10,555
1,1121146530,18829.3,8.074445,50,44,5,22,226.722,173,2,...,18,32,23,44,28,51,0,11470,1.130367e+10,555
2,1121150540,109.9,8.204151,67,99,3,12,366.090,407,0,...,22,36,26,39,35,68,0,12078,6.879562e+07,555
3,1121151290,265.1,9.421020,89,71,2,19,185.508,403,2,...,59,33,26,41,56,75,0,11736,8.293591e+07,555
4,1121151840,32034.9,9.312500,35,28,6,31,151.100,16,1,...,27,32,24,44,22,38,0,7784,1.263985e+10,555
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3491,8120253120,80185.4,6.915997,91,14,0,0,0.006,404,1,...,10,13,42,43,106,85,2,1660,4.468250e+07,948
3492,8120256970,111.5,5.486714,222,5,0,0,0.000,548,0,...,62,7,34,58,148,89,0,3379,2.679400e+05,948
3493,8120280400,5195.2,15.634457,128,28,0,0,3.149,409,11,...,88,10,37,52,122,91,0,2878,9.341507e+08,948
3494,8120295370,180.3,8.519233,33,22,0,0,5.139,587,1,...,0,10,45,45,133,89,0,64,6.378524e+07,948
